# 03 Describe sections

In [18]:
import matplotlib.pyplot as plt
import numpy as np

from os.path import exists, split, join
from os import makedirs
from tqdm import tqdm

import base.geometry as geom

from base.utils import list_all_centrelines, return_centreline_parametrisation



In [19]:
# change here the path to the repository of karst catalogue
data_repository = "F:/ScanLeica/data_paper/KarstConduitCatalogue" 
#data_repository = "D:/KarstConduitCatalogue" 

centreline_filepaths = list_all_centrelines(data_repository)

In [20]:
centreline_filepaths[11]

'F:\\ScanLeica\\data_paper\\KarstConduitCatalogue\\FauxMonnayeurs\\EntranceGallery\\geometric_descriptors\\FauxMonnayeurs_EntranceGallery_centreline.txt'

In [21]:
idx = 11
centreline_filepath = centreline_filepaths[idx]
assert exists(centreline_filepath)
centreline = np.loadtxt(centreline_filepath, skiprows=1)

In [22]:
sections_directory = join(split(centreline_filepath)[0], "sections")

In [23]:
interpolated_centreline, curvilinear_coords, normed_vectors = return_centreline_parametrisation(centreline, dt = 0.5)

In [24]:
sections = [np.loadtxt(join(sections_directory, f"s{c}.txt"), skiprows=1) for c in range(len(normed_vectors))]

In [ ]:
sections2d = [geom.to2D(section, nP) for section, nP in zip(sections, normed_vectors)]

In [ ]:
section_idx = 5 

In [ ]:
from sklearn.metrics import pairwise_distances

In [ ]:
with plt.xkcd():
    fig, ax = plt.subplots(figsize = (10,5))

    
    description = geom.describe(sections2d[section_idx])
    ax.plot(*sections2d[5].T, label = f"P = {description['perimeter']:.2f} m")
    ax.fill(*sections2d[5].T, label = f"A = {description['area']:.2f} m$^2$", alpha =.25)
    pwdist = pairwise_distances(sections2d[section_idx])
    start, end = np.unravel_index(pwdist.argmax(), pwdist.shape)
    L = np.max(pwdist)
    ax.plot(*sections2d[section_idx][[start, end]].T, label = f"L = {L:.2f} m")
    ax.set_xlim(-5, 10)
    

    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
        ax.set_aspect("equal")

   
    ax.legend(loc="upper right", frameon = False)
    
    ax.text(5, 1, f"hydraulic diameter: $\\frac{{4A}}{{P}}$={4*description['area']/description['perimeter']:.2f} m")
    ax.text(5, -0, f"isoperimetric quotient: $\\frac{{4\\pi A}}{{P^2}}$={description['isoperimetric_deficiency']:.2f}")
    ax.text(5, -1, f"gravelius compactness: $\\frac{{P}}{{2 \\sqrt{{\\pi A}}}}$ ={description['gravelius_compactness']:.2f}")
    ax.text(5, -2, f"morton spreading index: $\\frac{{4}}{{\\pi}} \\frac{{A}}{{L^2}}$={description['morton_spreading_index']:.2f}")


    plt.savefig("./temp/barcelona_figures/area_perimeter_section_figure.png")

    plt.show()

In [ ]:
with plt.xkcd():
    fig, ax = plt.subplots(figsize = (10,5))

    ax.plot(*sections2d[5].T)
    description = geom.describe(sections2d[5])
    ax.set_xlim(-5, 10)
    ax.scatter(*description["cog"], label="centre of gravity")

    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
        ax.set_aspect("equal")

    rmax, thearg = geom.compute_rmax(sections2d[5], description["cog"])
    ax.plot(*np.vstack([description["cog"],sections2d[5][thearg]]).T, color = "rebeccapurple", zorder = -10, label= "$r_{max}$", ls= "--")

    rmin, thearg = geom.compute_rmin(sections2d[5], description["cog"])
    ax.plot(*np.vstack([description["cog"],sections2d[5][thearg]]).T, color = "deeppink", zorder = -10, label= "$r_{min}$", ls = "--")
    

    ravg = description["r_average"]
    cog = description["cog"]
    thetai = np.linspace(0, 2*np.pi)
    ax.fill(cog[0]+ravg*np.cos(thetai), cog[1]+ravg*np.sin(thetai), color = "C1", alpha = 0.2, zorder = -100, label = "circle with $r_{average}$")
    ax.legend(loc="upper right", frameon = False)

    ax.text(5, 0, f"circularity 2: $\\frac{{rmin}}{{rmax}}$={rmin/rmax:.2f}")
    ax.text(5, -1, f"circularity 3: $\\frac{{rmax - rmin}}{{rmax}}$={(rmax -rmin)/rmax:.2f}")

    plt.savefig("./temp/barcelona_figures/radii_section_figure.png")

    plt.show()

In [ ]:
with plt.xkcd():
    fig, ax = plt.subplots(figsize = (10,5))

    ax.plot(*sections2d[5].T)
    description = geom.describe(sections2d[5])
    ax.set_xlim(-5, 10)

    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
        ax.set_aspect("equal")


    thetai = np.linspace(0, 2*np.pi)
    the_ellipse, ellipse_area, _, ellipse_aspect_ratio, ellipse_axes, ellipse_eigenvalues, ellipse_rotation = geom.fit_ellipse(sections2d[5])
    cog = description["cog"]

    ax.scatter(*cog, label="centre of gravity", zorder =100)
    print(ellipse_axes)
    ax.fill(*the_ellipse.T, color = "C1", alpha = 0.2, zorder = -100, label = "fitted ellipse")

    ax.plot(*np.vstack([cog,cog+ellipse_eigenvalues[0]*ellipse_axes[:, 0]]).T, color = "rebeccapurple", zorder = -10, label= "a", ls= "--")
    ax.plot(*np.vstack([cog,cog+ellipse_eigenvalues[1]*ellipse_axes[:, 1]]).T, color = "deeppink", label= "b", ls= "--", zorder = 2)

    ax.legend(loc="upper right", frameon = False)
    print(ellipse_rotation)
    ax.plot((cog[0], cog[0]+ellipse_eigenvalues[0]/1.618), (cog[1], cog[1]), color = "k", zorder = 0)
    smallthetai = np.linspace(0, -np.deg2rad(ellipse_rotation))
    ax.plot(cog[0]+ellipse_eigenvalues[0]/2*np.cos(smallthetai), cog[1]+ellipse_eigenvalues[0]/2*np.sin(smallthetai), color = "k", zorder = -1)
    
    ax.text(cog[0]+ellipse_eigenvalues[0]/1.6*np.cos(-np.deg2rad(ellipse_rotation)/2), 
            cog[1]+ellipse_eigenvalues[0]/1.6*np.sin(-np.deg2rad(ellipse_rotation)/2), 
            f"$\\theta$")
    ax.text(6, 0, f"$\\frac{{a}}{{b}}$={ellipse_eigenvalues[0]/ellipse_eigenvalues[1]:.2f}")
    ax.text(6, -1,
            f"$\\theta = {-ellipse_rotation:.1f}°$")
    plt.savefig("./temp/barcelona_figures/ellipse_section_figure.png")

    plt.show()

In [ ]:
# initialise a list of dictionaries 

descriptors = []

for section, curvilinear_coord in tqdm(zip(sections2d, curvilinear_coords), total=len(sections2d)):
    try:
        description = geom.describe(section)
        description["curvilinear_coord"] = curvilinear_coord

        descriptors.append(description)
    except ValueError:
        descriptors.append({})
        pass

In [ ]:
descriptors_filepath = join(split(centreline_filepath)[0], "section_descriptors.txt")

descriptor_keys = [elem for elem in descriptors[0].keys() if elem != "cog" and elem !="ellipse_area" and elem !="ellipse_axes" and elem !="ellipse_eigenvalues"]
m = len(descriptors)
n = len(descriptor_keys)
data = np.zeros((m,n))

header = ",".join(descriptor_keys)
print(header)
# cycle over each descriptor and store to numpy array. 
for j, key in enumerate(descriptor_keys):
    for i, elem in enumerate(descriptors):
        try:
            data[i, j] =  elem[key] 
        except KeyError:
            data[i, j] = np.nan
            pass
print(data)
# save the descriptor data 
np.savetxt(descriptors_filepath, data, header = header)

Now we can describe the rest of the sections using a filling up curve. 

In [ ]:
n = 50
fill_fractions = np.linspace(0, 1, n)

In [ ]:
for k, section2d in tqdm(enumerate(sections2d), total= len(sections2d)):
    
    areas = np.zeros(n)
    wetted_perimeters = np.zeros(n)

    for c,fill_fraction in enumerate(fill_fractions):
        areas[c] = geom.get_full_area(section2d, fill_height=fill_fraction)

        filled_areas = geom.return_below_fill_height(section2d, fill_height=fill_fraction)
        wetted_perimeter = 0
        
        if fill_fraction != 1:
            for filled_area in filled_areas:
                wetted_perimeter += geom.compute_ngon_wetted_perimeter(filled_area)
        else:
            wetted_perimeter = geom.compute_ngon_perimeter(section2d)

        wetted_perimeters[c] = wetted_perimeter
    
    data = np.vstack([fill_fractions, areas, wetted_perimeters]).T

    fill_curves_filepath = join(sections_directory, f"fill_curve_s{k}.txt")

    np.savetxt(fill_curves_filepath, data, header = "h, area, wetted_perimeter")

In [27]:
run_main_loop = True
n = 50
fill_fractions = np.linspace(0, 1, n)

if run_main_loop == True:

    for idx in range(len(descriptors_filepath)-1):
        try:
            
            centreline_filepath = centreline_filepaths[idx]
            assert exists(centreline_filepath)
            centreline = np.loadtxt(centreline_filepath, skiprows=1)

            sections_directory = join(split(centreline_filepath)[0], "sections")

            interpolated_centreline, curvilinear_coords, normed_vectors = return_centreline_parametrisation(centreline, dt=0.5)

            sections = [np.loadtxt(join(sections_directory, f"s{c}.txt"), skiprows=1) for c in range(len(normed_vectors))]

            sections2d = [geom.to2D(section, nP) for section, nP in zip(sections, normed_vectors)]

            # initialise a list of dictionaries 

            descriptors = []

            for section, curvilinear_coord in tqdm(zip(sections2d, curvilinear_coords), total=len(sections2d)):
                try:
                    description = geom.describe(section)
                    description["curvilinear_coord"] = curvilinear_coord

                    descriptors.append(description)
                except ValueError:
                    pass

            descriptors_filepath = join(split(centreline_filepath)[0], "section_descriptors.txt")

            descriptor_keys = [elem for elem in descriptors[0].keys() if elem != "cog" and elem !="ellipse_area" and elem !="ellipse_axes" and elem !="ellipse_eigenvalues"]
            m = len(descriptors)
            n = len(descriptor_keys)
            data = np.zeros((m,n))

            header = ",".join(descriptor_keys)
            # cycle over each descriptor and store to numpy array. 
            for j, key in enumerate(descriptor_keys):
                for i, elem in enumerate(descriptors):
                    try:
                        data[i, j] =  elem[key] 
                    except KeyError:
                        data[i, j] = np.nan
                        pass
            # save the descriptor data 
            np.savetxt(descriptors_filepath, data, header = header)
                        

            # for k, section2d in tqdm(enumerate(sections2d), total= len(sections2d)):
    
            #     areas = np.zeros(n)
            #     wetted_perimeters = np.zeros(n)

            #     for c,fill_fraction in enumerate(fill_fractions):
            #         areas[c] = geom.get_full_area(section2d, fill_height=fill_fraction)

            #         filled_areas = geom.return_below_fill_height(section2d, fill_height=fill_fraction)
            #         wetted_perimeter = 0
                    
            #         if fill_fraction != 1:
            #             for filled_area in filled_areas:
            #                 wetted_perimeter += geom.compute_ngon_wetted_perimeter(filled_area)
            #         else:
            #             wetted_perimeter = geom.compute_ngon_perimeter(section2d)

            #         wetted_perimeters[c] = wetted_perimeter
                
            #     data = np.vstack([fill_fractions, areas, wetted_perimeters]).T

            #     fill_curves_filepath = join(sections_directory, f"fill_curve_s{k}.txt")

            #     np.savetxt(fill_curves_filepath, data, header = "h, area, wetted_perimeter")

        except AssertionError:
            print(f"not all files exist yet for index {idx}")
            pass 

100%|████████████████████████████████████████████████████████████████████████████████| 435/435 [00:20<00:00, 21.68it/s]


IndexError: list index out of range